***EXPLORATORY DATA ANALYSIS***

In [ ]:
!pip install timm

In [ ]:
import os
import timm
import random
import ast
import gc
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import IPython.display as ipd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.ndimage import gaussian_filter1d
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
is_gpu = torch.cuda.is_available()
print(f"Using device: {device}")

base_path = '/kaggle/input/competitions/birdclef-2026'
train_csv_path = os.path.join(base_path, 'train.csv')
soundscapes_csv_path = os.path.join(base_path, 'train_soundscapes_labels.csv')
audio_dir = os.path.join(base_path, 'train_audio')
ss_audio_dir = os.path.join(base_path, 'train_soundscapes')

train_df = pd.read_csv(train_csv_path)
soundscapes_df = pd.read_csv(soundscapes_csv_path)

plt.figure(figsize=(14, 6))
top_species = train_df['primary_label'].value_counts().head(30)
sns.barplot(x=top_species.index, y=top_species.values, hue=top_species.index, legend=False, palette='viridis')
plt.xticks(rotation=45)
plt.title('Top 30 Species by Number of Recordings')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

if 'rating' in train_df.columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=train_df, x='rating', hue='rating', legend=False, palette='coolwarm')
    plt.title('Recording Ratings Distribution')
    plt.xlabel('Rating (0 = iNaturalist / Not Available)')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

soundscapes_df['label_count'] = soundscapes_df['primary_label'].apply(lambda x: len(str(x).split(';')))
label_dist = soundscapes_df['label_count'].value_counts().sort_index()

plt.figure(figsize=(8, 4))
sns.barplot(x=label_dist.index, y=label_dist.values, hue=label_dist.index, legend=False, palette='magma')
plt.title('Simultaneous Species in 5-second Soundscape Segments')
plt.xlabel('Number of Species')
plt.ylabel('Segment Count')
plt.tight_layout()
plt.show()

if 'latitude' in train_df.columns and 'longitude' in train_df.columns:
    plt.figure(figsize=(16, 8))
    
    geo_df = train_df.dropna(subset=['latitude', 'longitude'])
    
    plt.scatter(geo_df['longitude'], geo_df['latitude'], 
                alpha=0.3, s=8, c='darkcyan', edgecolors='none')
    
    plt.title('Distribuzione Geografica delle Registrazioni (BirdCLEF)')
    plt.xlabel('Longitudine')
    plt.ylabel('Latitudine')
    
    plt.xlim(-180, 180)
    plt.ylim(-90, 90)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

***EXTRACTION OF 5 SECONDS CHUNKS***

In [ ]:
# Librosa is used only for display/analysis here. 
# PyTorch/Torchaudio handles the actual pipeline in later blocks.
import librosa
import librosa.display

def extract_top_chunks(audio_path, sr=32000, duration=5.0, n_mels=128, max_chunks=3):
    """
    Extracts the highest-energy chunks from an audio file.
    """
    try:
        y, _ = librosa.load(audio_path, sr=sr)
    except Exception as e:
        print(f"Error loading {audio_path}: {e}")
        return [], []
        
    target_samples = int(sr * duration)
    total_samples = len(y)
    
    # Pad if audio is too short
    if total_samples <= target_samples:
        y_chunk = np.pad(y, (0, target_samples - total_samples))
        mel_spec = librosa.feature.melspectrogram(y=y_chunk, sr=sr, n_mels=n_mels, fmax=16000)
        # Using PCEN logic for visual consistency with our new architecture goals
        mel_spec = librosa.pcen(S=mel_spec * (2**31), sr=sr, hop_length=512, gain=0.98, bias=2, power=0.5, time_constant=0.4)
        return [y_chunk], [mel_spec]
        
    # Break into 5s non-overlapping chunks
    num_full_chunks = total_samples // target_samples
    chunks = []
    energies = []
    
    for i in range(num_full_chunks):
        start = i * target_samples
        end = start + target_samples
        chunk = y[start:end]
        rms = np.mean(librosa.feature.rms(y=chunk))
        chunks.append(chunk)
        energies.append(rms)
        
    # Handle remainder if it's longer than half the target duration
    remainder = total_samples % target_samples
    if remainder > (target_samples // 2):
        chunk = np.pad(y[-remainder:], (0, target_samples - remainder))
        rms = np.mean(librosa.feature.rms(y=chunk))
        chunks.append(chunk)
        energies.append(rms)
        
    # Get indices of the chunks with highest RMS energy
    top_indices = np.argsort(energies)[-max_chunks:][::-1]
    
    best_y = []
    best_mels = []
    for idx in top_indices:
        best_y.append(chunks[idx])
        mel_spec = librosa.feature.melspectrogram(y=chunks[idx], sr=sr, n_mels=n_mels, fmax=16000)
        
        # Apply PCEN for visualization instead of standard DB conversion
        pcen_spec = librosa.pcen(S=mel_spec * (2**31), sr=sr, hop_length=512, gain=0.98, bias=2, power=0.5, time_constant=0.4)
        best_mels.append(pcen_spec)
        
    return best_y, best_mels

sample_files = train_df.sample(2)

for _, row in sample_files.iterrows():
    audio_path = os.path.join(audio_dir, row['filename'])
    species = row['primary_label']
    
    # Check for secondary labels (Soft Targets context)
    secondary = row.get('secondary_labels', '[]')
    
    if os.path.exists(audio_path):
        print(f"\nPrimary Species: {species} | Secondary: {secondary} | File: {row['filename']}")
        
        y_chunks, pcen_specs = extract_top_chunks(audio_path, n_mels=128, max_chunks=3)
        
        for i, (y_chunk, pcen_spec) in enumerate(zip(y_chunks, pcen_specs)):
            plt.figure(figsize=(6, 2))
            # Display PCEN representation
            librosa.display.specshow(pcen_spec, sr=32000, x_axis='time', y_axis='mel', fmax=16000)
            plt.colorbar(format='%+2.0f')
            plt.title(f'PCEN Chunk {i+1} - {species}')
            plt.tight_layout()
            plt.show()
            
            display(ipd.Audio(y_chunk, rate=32000))
    else:
        print(f"File {audio_path} not found.")

***DATA PREPROCESSING AND EXTRACTION***

In [ ]:
# Handle singletons to prevent train_test_split crash
counts = train_df['primary_label'].value_counts()
singletons = counts[counts == 1].index
if len(singletons) > 0:
    print(f"Found {len(singletons)} singleton species! Duplicating...")
    df_singletons = train_df[train_df['primary_label'].isin(singletons)]
    train_df = pd.concat([train_df, df_singletons], ignore_index=True)

# Encode primary labels
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['primary_label'])
num_classes = len(label_encoder.classes_)

# Parse secondary labels from string representation to Python lists
train_df['secondary_labels_list'] = train_df['secondary_labels'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Cap samples at 150 per class to prevent heavy imbalance
df_balanced = train_df.groupby('primary_label').head(150).reset_index(drop=True)

class BirdWaveformDataset(Dataset):
    def __init__(self, df, audio_dir, label_encoder, target_sr=32000, duration=5.0, is_train=True):
        self.df = df
        self.audio_dir = audio_dir
        self.label_encoder = label_encoder
        self.num_classes = len(label_encoder.classes_)
        self.target_sr = target_sr
        self.target_samples = int(target_sr * duration)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.audio_dir, row['filename'])
        
        waveform, sr = torchaudio.load(file_path)
        
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        if sr != self.target_sr:
            resampler = T.Resample(sr, self.target_sr)
            waveform = resampler(waveform)
            
        # Dynamic Time Shifting
        if waveform.shape[1] > self.target_samples:
            if self.is_train:
                max_start = waveform.shape[1] - self.target_samples
                start = torch.randint(0, max_start, (1,)).item()
            else:
                start = 0
            waveform = waveform[:, start:start + self.target_samples]
            
        elif waveform.shape[1] < self.target_samples:
            pad_amount = self.target_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_amount))
            
        # Waveform Augmentation: Inject Gaussian White Noise randomly during training
        if self.is_train and torch.rand(1).item() < 0.5:
            noise_amplitude = 0.005 * torch.rand(1).item()
            waveform += torch.randn_like(waveform) * noise_amplitude
            
        # Soft Targets Implementation
        target = torch.zeros(self.num_classes, dtype=torch.float32)
        
        # Primary species gets full confidence (1.0)
        target[row['label_encoded']] = 1.0
        
        # Secondary species get partial confidence (0.3) to teach co-occurrence
        for sec_species in row['secondary_labels_list']:
            if sec_species in self.label_encoder.classes_:
                sec_idx = self.label_encoder.transform([sec_species])[0]
                target[sec_idx] = 0.3
                
        return waveform, target

# Split maintaining class stratification based on primary label
train_df_split, val_df_split = train_test_split(
    df_balanced, test_size=0.2, stratify=df_balanced['label_encoded'], random_state=42
)

train_dataset = BirdWaveformDataset(train_df_split, audio_dir, label_encoder, is_train=True)
val_dataset = BirdWaveformDataset(val_df_split, audio_dir, label_encoder, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

print(f"DATASETS READY")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

***MODEL DEFINITION***

In [ ]:
class AudioToSpectrogramGPU(nn.Module):
    def __init__(self, sr=32000, n_mels=256, n_fft=2048, hop_length=512, f_min=40, f_max=15000):
        super(AudioToSpectrogramGPU, self).__init__()
        
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            f_min=f_min,
            f_max=f_max,
            n_mels=n_mels,
            power=2.0 
        )
        
        # PCEN hyperparameters
        self.eps = 1e-6
        self.s = 0.025
        self.alpha = 0.98
        self.delta = 2.0
        self.r = 0.5
        
        # SpecAugment masks
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=36)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=64)

    def forward(self, waveform):
        x = self.mel_spec(waveform)
        
        ema = x.clone()
        for t in range(1, x.size(-1)):
            ema[..., t] = (1 - self.s) * ema[..., t - 1] + self.s * x[..., t]
            
        x = (x / (self.eps + ema)**self.alpha + self.delta)**self.r - self.delta**self.r
        x = (x - x.mean()) / (x.std() + 1e-6)
        
        if self.training:
            x = self.freq_mask(x)
            x = self.time_mask(x)
            
        return x

class AttentivePooling(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(AttentivePooling, self).__init__()
        self.attention = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)
        self.classifier = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)

    def forward(self, x):
        att_weights = torch.softmax(self.attention(x), dim=-1)
        frame_logits = self.classifier(x)
        clip_logits = torch.sum(att_weights * frame_logits, dim=-1)
        return clip_logits, frame_logits

class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss) 
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

def apply_mixup(waveforms, labels):
    lam = np.random.beta(0.5, 0.5) if np.random.rand() < 0.5 else 1.0
    if lam == 1.0: return waveforms, labels
    
    indices = torch.randperm(waveforms.size(0)).to(waveforms.device)
    shuffled_waveforms = waveforms[indices]
    shuffled_labels = labels[indices]
    
    mixed_waveforms = lam * waveforms + (1.0 - lam) * shuffled_waveforms
    mixed_labels = torch.max(labels, shuffled_labels)
    
    return mixed_waveforms, mixed_labels

class BirdSED_Pretrained(nn.Module):
    def __init__(self, backbone_name='tf_efficientnet_b2', num_classes=264, pretrained=True):
        super(BirdSED_Pretrained, self).__init__()
        
        self.audio_extractor = AudioToSpectrogramGPU()
        
        self.backbone = timm.create_model(
            model_name=backbone_name, 
            pretrained=pretrained, 
            in_chans=1, 
            num_classes=0,
            global_pool='' 
        )
        
        in_features = self.backbone.num_features
        
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.dropout = nn.Dropout(0.5)
        self.sed_head = AttentivePooling(in_channels=in_features, num_classes=num_classes)

    def forward(self, waveform):
        x = self.audio_extractor(waveform)
        x = self.backbone.forward_features(x)
        x = self.freq_pool(x).squeeze(2)
        x = self.dropout(x)
        clip_logits, _ = self.sed_head(x)
        return clip_logits

In [ ]:
def time_to_seconds(t_str):
    h, m, s = map(int, str(t_str).split(':'))
    return h * 3600 + m * 60 + s

def parse_labels(val):
    if pd.isna(val): return []
    return [str(l) for l in re.split(r'[;,\s]+', str(val).strip()) if l]

class SoundscapeWaveformDataset(Dataset):
    def __init__(self, df, audio_dir, label_encoder, target_sr=32000, duration=5.0):
        self.df = df
        self.audio_dir = audio_dir
        self.label_encoder = label_encoder
        self.num_classes = len(label_encoder.classes_)
        self.target_sr = target_sr
        self.frames_per_chunk = int(target_sr * duration)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.audio_dir, row['filename'])
        frame_offset = int(row['start_sec'] * self.target_sr)
        
        try:
            waveform, sr = torchaudio.load(file_path, frame_offset=frame_offset, num_frames=self.frames_per_chunk)
        except Exception:
            waveform = torch.zeros((1, self.frames_per_chunk))
            sr = self.target_sr
            
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        if waveform.shape[1] < self.frames_per_chunk:
            waveform = torch.nn.functional.pad(waveform, (0, self.frames_per_chunk - waveform.shape[1]))

        target = torch.zeros(self.num_classes, dtype=torch.float32)
        for species in row['primary_label']:
            try:
                class_idx = self.label_encoder.transform([species])[0]
                target[class_idx] = 1.0
            except ValueError:
                pass
                
        return waveform, target

df_ss_full = pd.read_csv(soundscapes_csv_path)
df_ss_full['start_sec'] = df_ss_full['start'].apply(time_to_seconds)
df_ss_full['primary_label'] = df_ss_full['primary_label'].apply(parse_labels)

ss_dataset_full = SoundscapeWaveformDataset(df_ss_full, ss_audio_dir, label_encoder)

train_size = int(0.8 * len(ss_dataset_full))
val_size = len(ss_dataset_full) - train_size
train_ss, val_ss = torch.utils.data.random_split(
    ss_dataset_full, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_ss_loader = DataLoader(train_ss, batch_size=32, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_ss_loader = DataLoader(val_ss, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

def train_pretrained_pipeline(backbone_name, save_prefix):
    print(f"Pipeline start: {backbone_name}")
    
    model = BirdSED_Pretrained(backbone_name=backbone_name, num_classes=len(label_encoder.classes_)).to(device)
    
    epochs_a = 15 if is_gpu else 1
    epochs_b = 10 if is_gpu else 1
    THRESHOLD = 0.3
    
    save_path_A = f'{save_prefix}_PhaseA.pth'
    save_path_B = f'{save_prefix}_PhaseB.pth'
    

    print(f"Phase A ({backbone_name})")
    criterion = FocalLoss(alpha=1.0, gamma=2.0)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4) # Lower lr for transfer learning
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader), 
        epochs=epochs_a, pct_start=0.2, div_factor=10.0
    )
    
    best_val_loss_A = float('inf')
    
    for epoch in range(epochs_a):
        model.train()
        train_loss = 0.0
        train_bar = tqdm(train_loader, desc=f"A-Train [Ep {epoch+1}/{epochs_a}]")
        for waveforms, labels in train_bar:
            waveforms, labels = waveforms.to(device), labels.to(device)
            waveforms, labels = apply_mixup(waveforms, labels)
            
            optimizer.zero_grad()
            logits = model(waveforms)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step() 
            
            train_loss += loss.item()
            train_bar.set_postfix(FLoss=f"{loss.item():.4f}")
            if not is_gpu: break

        model.eval()
        val_loss, all_preds, all_true, all_probs = 0.0, [], [], []
        with torch.no_grad():
            for waveforms, labels in val_loader:
                waveforms, labels = waveforms.to(device), labels.to(device)
                logits = model(waveforms)
                loss = criterion(logits, labels)
                val_loss += loss.item()
                
                probs = torch.sigmoid(logits)
                preds = (probs > THRESHOLD).int()
                
                all_probs.extend(probs.cpu().numpy()) 
                all_preds.extend(preds.cpu().numpy())
                all_true.extend((labels > 0.5).int().cpu().numpy())
                if not is_gpu: break

        if is_gpu:
            avg_val_loss = val_loss / len(val_loader)
            print(f"-> A-Ep [{epoch+1}/{epochs_a}] Val FLoss: {avg_val_loss:.4f}")
            
            if avg_val_loss < best_val_loss_A:
                best_val_loss_A = avg_val_loss
                torch.save(model.state_dict(), save_path_A)
                
        gc.collect()

    print(f"\nPhase B ({backbone_name})")
    model.load_state_dict(torch.load(save_path_A))
    
    optimizer_b = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler_b = optim.lr_scheduler.OneCycleLR(
        optimizer_b, max_lr=1e-4, steps_per_epoch=len(train_ss_loader), 
        epochs=epochs_b, pct_start=0.2, div_factor=10.0
    )
    
    best_val_loss_B = float('inf')
    
    for epoch in range(epochs_b):
        model.train()
        train_loss = 0.0
        train_bar = tqdm(train_ss_loader, desc=f"B-Train [Ep {epoch+1}/{epochs_b}]")
        for waveforms, labels in train_bar:
            waveforms, labels = waveforms.to(device), labels.to(device)
            waveforms, labels = apply_mixup(waveforms, labels)
            
            optimizer_b.zero_grad()
            logits = model(waveforms)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_b.step()
            scheduler_b.step()
            train_loss += loss.item()
            train_bar.set_postfix(FLoss=f"{loss.item():.4f}")
            if not is_gpu: break
            
        model.eval()
        val_loss, all_preds, all_true = 0.0, [], []
        with torch.no_grad():
            for waveforms, labels in val_ss_loader:
                waveforms, labels = waveforms.to(device), labels.to(device)
                logits = model(waveforms)
                loss = criterion(logits, labels)
                val_loss += loss.item()
                
                probs = torch.sigmoid(logits)
                preds = (probs > THRESHOLD).int()
                
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(labels.cpu().numpy())
                if not is_gpu: break

        if is_gpu:
            avg_val_loss = val_loss / len(val_ss_loader)
            print(f"-> B-Ep [{epoch+1}/{epochs_b}] Val FLoss: {avg_val_loss:.4f}")
            
            if avg_val_loss < best_val_loss_B:
                best_val_loss_B = avg_val_loss
                torch.save(model.state_dict(), save_path_B)
                
        gc.collect()
        
    print(f"Pipeline completed {backbone_name}.")

In [ ]:
train_pretrained_pipeline(backbone_name='tf_efficientnet_b2', save_prefix='sed_effnetb2')

train_pretrained_pipeline(backbone_name='convnext_tiny', save_prefix='sed_convnext')